# 🚀 AeroGuard TSLM — Notebook 3: Model Architecture & Inference Pre-Check

This notebook tests loading the trained **AeroGuard TSLM** multimodal model, verifying the **TimeSeriesPatchEncoder**, running a forward pass on held-out test telemetry, and comparing the scalar RUL head output against the ground truth.

### Objectives:
1. Verify checkpoint files in `models/aeroguard_tslm/`.
2. Instantiate `AeroGuardTSLM` (`SmolLM-135M-Instruct` backbone + LoRA + patch encoder).
3. Run a forward pass on a held-out test engine window (e.g. Engine #84).
4. Evaluate both the auxiliary scalar RUL prediction and the generated Chain-of-Thought diagnostics.
5. Test physical fault isolation logic (`training.fault_isolation`).

In [ ]:
import sys
from pathlib import Path
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from training.dataset_loader import CMAPSSCoTDataset, NormalizationStats
from training.train_opentslm import AeroGuardTSLM
from training.fault_isolation import isolate_component_fault

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✔ Using compute device: {device}")

## 1. Verify Checkpoint Artifacts

Check whether the model weights and adapter files are available in `models/aeroguard_tslm/`.

In [ ]:
MODEL_DIR = PROJECT_ROOT / "models" / "aeroguard_tslm"

required_files = [
    MODEL_DIR / "preprocessing.json",
    MODEL_DIR / "tslm_adapters.pt",
    MODEL_DIR / "lora_adapters" / "adapter_config.json",
    MODEL_DIR / "lora_adapters" / "adapter_model.safetensors",
]

for p in required_files:
    exists = p.exists()
    status = "✔ Found" if exists else "❌ Missing"
    print(f"{status}: {p.relative_to(PROJECT_ROOT)}")

## 2. Load Model & Trained Weights

Initialize the multimodal architecture and load the fine-tuned LoRA adapters and patch encoder.

In [ ]:
from training.inference import AeroGuardPredictor
predictor = AeroGuardPredictor(model_dir=MODEL_DIR, device=device)
model = predictor.model
print("Model and saved adapters loaded.")


## 3. Load Held-Out Test Window (Engine #84)

Let's load a 30-cycle telemetry window from strictly held-out **Engine Unit #84** (unseen during training) near failure threshold.

In [ ]:
WINDOWS_PATH = PROJECT_ROOT / "data" / "processed" / "windows.jsonl"
NORM_PATH = MODEL_DIR / "preprocessing.json"

test_dataset = CMAPSSCoTDataset(
    jsonl_path=str(WINDOWS_PATH),
    split="test",
    normalization=NormalizationStats.load(NORM_PATH),
    include_targets=True
)

# Find a near-failure window for Engine #84
sample = None
for i in range(len(test_dataset)):
    item = test_dataset[i]
    if item["unit_number"] == 84 and item["rul"] <= 20:
        sample = item
        break

assert sample is not None, "Sample window not found."
print(f"• Selected Window:    {sample['record_id']}")
print(f"• Engine Unit:        #{sample['unit_number']}")
print(f"• Inspection Cycle:   {sample['cycle']}")
print(f"• True Remaining RUL: {sample['rul']} cycles")

## 4. Run Multimodal Inference (RUL & CoT Generation)

Now pass the normalized $[14, 30]$ sensor tensor into AeroGuard TSLM to predict scalar RUL and generate natural language diagnostics.

In [ ]:
sensor_tensor = sample["sensor_series"].to(device) # Shape: [14, 30]

# Generate assessment
pred_rul, diagnostic_text = model.generate_assessment(
    sensor_series=sensor_tensor,
    prompt_text=sample["prompt"],
    max_new_tokens=128
)

print("=" * 65)
print("✈️ AEROGUARD TSLM INFERENCE RESULT")
print("=" * 65)
print(f"• True Remaining Useful Life:      {sample['rul']} cycles")
print(f"• Predicted Remaining Useful Life: {pred_rul:.1f} cycles")
print(f"• Absolute Error:                  {abs(pred_rul - sample['rul']):.1f} cycles")
print("=" * 65)
print("\nGenerated Chain-of-Thought Diagnostics:")
print("-" * 65)
print(diagnostic_text)

## 5. Physical Component Fault Isolation

Test the deterministic fault isolation module to pinpoint the Line-Replaceable Unit (LRU) part and AMM task card based on aerothermal drift.

In [ ]:
import json
with WINDOWS_PATH.open() as handle:
    raw_record = next(json.loads(line) for line in handle
                      if json.loads(line)["record_id"] == sample["record_id"])
fault_report = isolate_component_fault(raw_record["series"], pred_rul)
print("=== Rule-Based Component Assessment ===")
print(f"Station: {fault_report.station_name}")
print(f"Module: {fault_report.module_name}")
print(f"Status: {fault_report.status}")
print(f"Suggested action: {fault_report.action_summary}")
